## 3 - Exploratory Analysis
ეს ნოუთბუქი ექვემდებარება მონაცემების შესწავლას, რათა გამოვიკვლიოთ თუ რა ინფორმაციის მიღბა შეგვიძლია მოცემული data-თი


In [0]:
%sql
SELECT
    YEAR(Order_Date)                        AS Year,
    ROUND(SUM(Order_Amount), 2)             AS Total_Revenue,
    ROUND(SUM(Profit_Amount), 2)            AS Total_Profit,
    ROUND(AVG(Profit_Margin_Percent), 2)    AS Avg_Margin_Pct,
    COUNT(Order_ID)                AS Total_Orders,
    COUNT(DISTINCT Customer_ID)             AS Unique_Customers
FROM getdata.calculated.orders
GROUP BY YEAR(Order_Date)
ORDER BY Year

Year,Total_Revenue,Total_Profit,Avg_Margin_Pct,Total_Orders,Unique_Customers
2023,2962427.11,581977.99,21.99,7577,5152
2024,2797262.51,543446.89,22.1,7454,5027
2025,2792985.7,547851.06,22.0,7520,5117
2026,2817368.67,542640.33,21.79,7449,5069


In [0]:
%sql
SELECT
    Product_Category,
    COUNT(Order_ID)                                                        AS Total_Orders,
    SUM(CASE WHEN Returned = TRUE THEN 1 ELSE 0 END)                       AS Returned_Orders,
    ROUND(SUM(CASE WHEN Returned = TRUE THEN 1 ELSE 0 END) * 100.0
        / COUNT(Order_ID), 2)                                              AS Return_Rate_Pct,
    SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1 ELSE 0 END)            AS Cancelled_Orders,
    ROUND(SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1 ELSE 0 END) * 100.0
        / COUNT(Order_ID), 2)                                              AS Cancellation_Rate_Pct,
    ROUND(SUM(CASE WHEN Returned = TRUE THEN Profit_Amount ELSE 0 END), 2) AS Profit_Lost_To_Returns
FROM getdata.calculated.orders
GROUP BY Product_Category
ORDER BY Return_Rate_Pct DESC

Product_Category,Total_Orders,Returned_Orders,Return_Rate_Pct,Cancelled_Orders,Cancellation_Rate_Pct,Profit_Lost_To_Returns
Fashion,6092,845,13.87,334,5.48,50543.3
Books,2163,236,10.91,137,6.33,4239.52
Sports,2439,239,9.80,147,6.03,14854.42
Electronics,4699,432,9.19,284,6.04,96261.97
Groceries,5380,475,8.83,322,5.99,6548.42
Toys,2062,182,8.83,103,5.00,5490.83
Beauty,2971,261,8.78,181,6.09,11141.02
Home & Kitchen,4194,363,8.66,263,6.27,33792.35


In [0]:
%sql
SELECT
    CASE
        WHEN Discount_Percent = 0          THEN '0% - No Discount'
        WHEN Discount_Percent <= 10        THEN '1-10%'
        WHEN Discount_Percent <= 20        THEN '11-20%'
        WHEN Discount_Percent <= 30        THEN '21-30%'
        ELSE '30%+'
    END                                               AS Discount_Bucket,
    COUNT(Order_ID)                                   AS Total_Orders,
    ROUND(AVG(Profit_Margin_Percent), 2)              AS Avg_Margin_Pct,
    ROUND(AVG(Order_Amount), 2)                       AS Avg_Order_Amount,
    ROUND(SUM(CASE WHEN Returned = TRUE THEN 1 ELSE 0 END) * 100.0
        / COUNT(Order_ID), 2)                          AS Return_Rate_Pct,
    ROUND(SUM(Profit_Amount), 2)                      AS Total_Profit
FROM getdata.calculated.orders
GROUP BY Discount_Bucket
ORDER BY Discount_Bucket

Discount_Bucket,Total_Orders,Avg_Margin_Pct,Avg_Order_Amount,Return_Rate_Pct,Total_Profit
0% - No Discount,7882,24.99,427.84,10.33,750243.74
1-10%,8373,23.16,393.49,10.06,673263.73
11-20%,7768,20.71,363.83,10.18,506446.16
21-30%,4184,18.68,320.17,9.92,213257.29
30%+,1793,16.24,299.68,9.54,72705.35


In [0]:
%sql
SELECT
    Customer_Segment,
    Membership_Status,
    COUNT(DISTINCT Customer_ID)               AS Unique_Customers,
    COUNT(Order_ID)                           AS Total_Orders,
    ROUND(AVG(Customer_Lifetime_Value), 2)    AS Avg_CLV,
    ROUND(AVG(Order_Amount), 2)               AS Avg_Order_Amount,
    ROUND(SUM(CASE WHEN Returned = TRUE THEN 1 ELSE 0 END) * 100.0
        / COUNT(Order_ID), 2)                  AS Return_Rate_Pct,
    ROUND(AVG(Review_Rating), 2)              AS Avg_Review_Rating
FROM getdata.calculated.orders
GROUP BY
    Customer_Segment,
    Membership_Status
ORDER BY
    Avg_CLV DESC

Customer_Segment,Membership_Status,Unique_Customers,Total_Orders,Avg_CLV,Avg_Order_Amount,Return_Rate_Pct,Avg_Review_Rating
Premium,Silver,694,712,5653.69,398.82,9.27,4.06
Premium,Platinum,203,204,5456.06,399.84,6.37,4.26
Premium,Standard,1485,1606,5322.3,378.97,10.15,4.04
Premium,Gold,467,483,5140.83,354.02,7.87,4.09
Loyal,Platinum,380,397,4862.03,367.67,14.11,4.18
Loyal,Silver,1596,1756,4724.78,379.07,9.74,4.04
New,Platinum,439,450,4678.12,442.06,12.44,4.2
Loyal,Gold,1119,1186,4594.32,366.06,10.54,4.01
Loyal,Standard,3165,3875,4567.39,364.87,9.57,4.06
Returning,Silver,2467,2880,4256.54,392.07,8.85,4.04


In [0]:
%sql
-- First order date per customer = when they were "acquired"
WITH FirstOrder AS (
    SELECT
        Customer_ID,
        MIN(Order_Date) AS First_Order_Date
    FROM getdata.calculated.orders
    GROUP BY Customer_ID
)

SELECT
    YEAR(First_Order_Date)                  AS Acquisition_Year,
    MONTH(First_Order_Date)                 AS Acquisition_Month,
    COUNT(Customer_ID)                      AS New_Customers
FROM FirstOrder
GROUP BY
    YEAR(First_Order_Date),
    MONTH(First_Order_Date)
ORDER BY
    Acquisition_Year,
    Acquisition_Month

Acquisition_Year,Acquisition_Month,New_Customers
2023,1,686
2023,2,521
2023,3,549
2023,4,446
2023,5,476
2023,6,433
2023,7,388
2023,8,400
2023,9,332
2023,10,303


In [0]:
%sql
SELECT
    Traffic_Source,
    COUNT(DISTINCT Customer_ID)               AS Unique_Customers,
    COUNT(Order_ID)                           AS Total_Orders,
    ROUND(AVG(Order_Amount), 2)               AS Avg_Order_Amount,
    ROUND(AVG(Customer_Lifetime_Value), 2)    AS Avg_CLV,
    ROUND(SUM(CASE WHEN Returned = TRUE THEN 1 ELSE 0 END) * 100.0
        / COUNT(Order_ID), 2)                  AS Return_Rate_Pct,
    ROUND(SUM(Profit_Amount), 2)              AS Total_Profit
FROM getdata.calculated.orders
GROUP BY Traffic_Source
ORDER BY Avg_CLV DESC

Traffic_Source,Unique_Customers,Total_Orders,Avg_Order_Amount,Avg_CLV,Return_Rate_Pct,Total_Profit
Social Media,3887,5059,391.64,4485.12,10.34,386217.67
Direct,3897,5073,384.0,4408.74,9.50,385411.44
Email,3809,4949,384.54,4384.02,10.28,376196.97
Paid Ads,3723,4811,381.3,4356.36,10.04,349578.38
Organic Search,3879,5098,367.92,4300.67,10.34,355492.41
Referral,3887,5010,364.78,4298.33,10.16,363019.4
